# 02 — Model Training: Credit Risk GBM

Train a HistGradientBoostingClassifier on the Gold feature set with MLflow experiment tracking.  
Evaluates on validation and hold-out test sets. Computes AUC, KS, Gini, and decile analysis.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve

plt.style.use('seaborn-v0_8-whitegrid')

GOLD_DIR = Path("../data/gold")
MODELS_DIR = Path("../data/models")

with open(GOLD_DIR / "feature_metadata.json") as f:
    meta = json.load(f)
feature_cols = meta["feature_columns"]

train = pd.read_parquet(GOLD_DIR / "features_train.parquet")
val = pd.read_parquet(GOLD_DIR / "features_val.parquet")
test = pd.read_parquet(GOLD_DIR / "features_test.parquet")

X_train, y_train = train[feature_cols], train["default"]
X_val, y_val = val[feature_cols], val["default"]
X_test, y_test = test[feature_cols], test["default"]

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 1. Train Model

In [ ]:
# Combine train+val for early stopping with validation_fraction
combined_X = pd.concat([X_train, X_val], ignore_index=True)
combined_y = pd.concat([y_train, y_val], ignore_index=True)
val_fraction = len(X_val) / len(combined_X)

model = HistGradientBoostingClassifier(
    max_iter=1000,
    max_depth=6,
    learning_rate=0.05,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=1.0,
    max_bins=255,
    early_stopping=True,
    validation_fraction=val_fraction,
    n_iter_no_change=50,
    scoring="roc_auc",
    random_state=42,
    verbose=1,
)

model.fit(combined_X, combined_y)
print(f"\nBest iteration: {model.n_iter_}")

## 2. Evaluation — ROC, Precision-Recall, Calibration

In [ ]:
def compute_ks(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for X, y, name, color in [(X_train, y_train, 'Train', '#2ecc71'), 
                            (X_val, y_val, 'Val', '#f39c12'),
                            (X_test, y_test, 'Test', '#e74c3c')]:
    scores = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, scores)
    ks = compute_ks(y, scores)
    
    # ROC
    fpr, tpr, _ = roc_curve(y, scores)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
    
    # Precision-Recall
    prec, rec, _ = precision_recall_curve(y, scores)
    axes[1].plot(rec, prec, color=color, label=f'{name}')
    
    # Calibration
    prob_true, prob_pred = calibration_curve(y, scores, n_bins=10)
    axes[2].plot(prob_pred, prob_true, 's-', color=color, label=f'{name}')
    
    print(f"{name:6s} | AUC={auc:.4f}  KS={ks:.4f}  Gini={2*auc-1:.4f}")

axes[0].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

axes[2].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[2].set_xlabel('Predicted Probability'); axes[2].set_ylabel('Actual Probability')
axes[2].set_title('Calibration Curve'); axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Decile Analysis — Risk Rank Ordering

In [ ]:
# Decile analysis on test set
test_scores = model.predict_proba(X_test)[:, 1]
decile_df = pd.DataFrame({'score': test_scores, 'default': y_test.values})
decile_df['decile'] = pd.qcut(decile_df['score'], 10, labels=range(1, 11))

decile_stats = decile_df.groupby('decile', observed=True).agg(
    count=('default', 'size'),
    n_defaults=('default', 'sum'),
    default_rate=('default', 'mean'),
    avg_score=('score', 'mean'),
    min_score=('score', 'min'),
    max_score=('score', 'max'),
).reset_index()

decile_stats['cumulative_defaults'] = decile_stats['n_defaults'].cumsum()
decile_stats['capture_rate'] = decile_stats['cumulative_defaults'] / decile_stats['n_defaults'].sum()

print(decile_stats.to_string(index=False))

# Plot
fig, ax1 = plt.subplots(figsize=(12, 6))
bars = ax1.bar(decile_stats['decile'].astype(str), decile_stats['default_rate'],
               color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)), edgecolor='black')
ax1.set_xlabel('Score Decile (1=lowest risk)')
ax1.set_ylabel('Default Rate', color='black')
ax1.set_title('Test Set: Default Rate & Cumulative Capture by Score Decile')

ax2 = ax1.twinx()
ax2.plot(decile_stats['decile'].astype(str), decile_stats['capture_rate'], 
         'ko-', linewidth=2, markersize=8, label='Cumulative Capture')
ax2.set_ylabel('Cumulative Default Capture Rate', color='black')
ax2.legend(loc='center right')

plt.tight_layout()
plt.show()

print(f"\nTop 3 deciles capture {decile_stats.iloc[-3:]['n_defaults'].sum() / decile_stats['n_defaults'].sum():.1%} of all defaults")